# 3. Cross-Method QC & Comparison

**Purpose:** Compare segmentation methods side-by-side on the same sample.  
**Requires:** At least 2 methods to have completed SLURM jobs.  
**Bonus:** If the QC SLURM job ran, load its pre-computed comparison table.

Metrics compared:
- Cell counts, gene detection, count distributions
- Cell morphology (area) if available
- Cluster purity (if reference annotations exist)
- Spatial concordance between methods

In [ ]:
import os
import json
import yaml
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
from pathlib import Path

sc.settings.set_figure_params(dpi=100, frameon=False)
%matplotlib inline

In [ ]:
CONFIG_PATH = "../config/pipeline_config.yaml"
with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)

SAMPLE_ID = cfg["sample"]["id"]
BASE_OUT = Path(cfg["paths"]["base_output_dir"]) / SAMPLE_ID

# Load all available h5ad files
METHODS = ["proseg", "baysor", "cellpose", "bidcell"]
adatas = {}
for m in METHODS:
    h5ad = BASE_OUT / m / f"{SAMPLE_ID}.h5ad"
    if h5ad.exists():
        adata = sc.read_h5ad(h5ad)
        sc.pp.calculate_qc_metrics(adata, inplace=True)
        adatas[m] = adata

print(f"Loaded {len(adatas)} methods: {list(adatas.keys())}")

## Summary table

In [ ]:
# Check if QC job already produced a comparison
comparison_csv = BASE_OUT / "comparison" / "method_comparison.csv"
if comparison_csv.exists():
    comparison = pd.read_csv(comparison_csv)
    print("Loaded pre-computed comparison from QC job:")
    display(comparison)
else:
    # Compute on the fly
    rows = []
    for method, adata in adatas.items():
        counts = np.asarray(adata.X.sum(axis=1)).flatten()
        genes = np.asarray((adata.X > 0).sum(axis=1)).flatten()
        rows.append({
            "method": method,
            "n_cells": adata.n_obs,
            "n_genes": adata.n_vars,
            "median_counts": np.median(counts),
            "median_genes": np.median(genes),
            "pct_lt_10_counts": round(np.mean(counts < 10) * 100, 1),
            "pct_lt_5_genes": round(np.mean(genes < 5) * 100, 1),
        })
    comparison = pd.DataFrame(rows)
    display(comparison)

## Distribution comparisons

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
colors = plt.cm.Set2(np.linspace(0, 1, len(adatas)))

for i, (method, adata) in enumerate(adatas.items()):
    # Total counts
    axes[0].hist(
        adata.obs["total_counts"], bins=50, alpha=0.5,
        label=f"{method} (n={adata.n_obs:,})", color=colors[i],
    )
    # Genes per cell
    axes[1].hist(
        adata.obs["n_genes_by_counts"], bins=50, alpha=0.5,
        label=method, color=colors[i],
    )
    # % top 50 genes
    if "pct_counts_in_top_50_genes" in adata.obs:
        axes[2].hist(
            adata.obs["pct_counts_in_top_50_genes"], bins=50, alpha=0.5,
            label=method, color=colors[i],
        )

axes[0].set_xlabel("Total Counts per Cell")
axes[0].set_ylabel("Number of Cells")
axes[0].legend()
axes[1].set_xlabel("Genes Detected per Cell")
axes[1].legend()
axes[2].set_xlabel("% Counts in Top 50 Genes")
axes[2].legend()

fig.suptitle(f"Method Comparison — {SAMPLE_ID}", fontsize=14)
plt.tight_layout()
plt.show()

## Cell area comparison (if available)

In [ ]:
area_data = []
for method, adata in adatas.items():
    if "area" in adata.obs.columns:
        area_data.append(pd.DataFrame({"method": method, "area": adata.obs["area"]}))

if area_data:
    area_df = pd.concat(area_data)
    fig, ax = plt.subplots(figsize=(8, 5))
    area_df.boxplot(column="area", by="method", ax=ax, showfliers=False)
    ax.set_title(f"Cell Area by Method — {SAMPLE_ID}")
    ax.set_xlabel("Method")
    ax.set_ylabel("Cell Area")
    plt.suptitle("")  # Remove auto title
    plt.tight_layout()
    plt.show()
else:
    print("No cell area information available in any method's output.")

## Runtime comparison

In [ ]:
runtimes = []
for method in adatas:
    meta_path = BASE_OUT / method / f"run_metadata_{method}.json"
    if meta_path.exists():
        with open(meta_path) as f:
            meta = json.load(f)
        runtimes.append({"method": method, "minutes": meta.get("elapsed_minutes", 0)})

if runtimes:
    rt_df = pd.DataFrame(runtimes)
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.barh(rt_df["method"], rt_df["minutes"], color="steelblue")
    ax.set_xlabel("Runtime (minutes)")
    ax.set_title("Segmentation Runtime")
    for i, row in rt_df.iterrows():
        ax.text(row["minutes"] + 0.5, i, f"{row['minutes']:.0f}m", va="center")
    plt.tight_layout()
    plt.show()

## Gene overlap across methods
How consistent are the detected gene panels?

In [ ]:
gene_sets = {m: set(a.var_names) for m, a in adatas.items()}

methods_list = list(gene_sets.keys())
if len(methods_list) >= 2:
    shared = gene_sets[methods_list[0]]
    for m in methods_list[1:]:
        shared = shared & gene_sets[m]
    
    print(f"Genes shared across all methods: {len(shared)}")
    for m, gs in gene_sets.items():
        unique = gs - shared
        print(f"  {m}: {len(gs)} total, {len(unique)} unique")

## Cluster consistency (optional)

If you have reference cell type annotations, you can assess cluster purity here.  
This requires running each method's results through a standard annotation pipeline first.

In [ ]:
# Placeholder for cluster purity analysis
# To use:
# 1. Run clustering + annotation on each method's h5ad (e.g., via celltypist, scANVI, or manual)
# 2. Store annotations in adata.obs["cell_type"]
# 3. Compare consistency across methods

annotated = {m: a for m, a in adatas.items() if "cell_type" in a.obs.columns}
if annotated:
    for method, adata in annotated.items():
        print(f"\n{method} cell types:")
        print(adata.obs["cell_type"].value_counts().head(10))
else:
    print("No cell type annotations found yet.")
    print("Run annotation on each method's h5ad, then re-run this section.")

---

## CellSPA QC (R-based)

For deeper QC using CellSPA, use the R magic cells below or run the R script separately.

In [ ]:
# Uncomment to use R magic:
# from rpy2.robjects import r, pandas2ri
# %load_ext rpy2.ipython

In [ ]:
# %%R
# library(CellSPA)
# # Load data and run CellSPA QC here
# # See CellSPA docs: https://github.com/SydneyBioX/CellSPA